<a href="https://colab.research.google.com/github/vrishinm/Data3001-Assignment-Group-2/blob/main/section2%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA3001 — Agulhas region: data availability

Counts for Section 2 of the project proposal.
Region R = [10°E, 45°E] × [45°S, 20°S].

Data: NOAA Global Drifter Program hourly product, version 2.01.1,
accessed via CloudDrift on [23/09/2026]. DOI: 10.25921/x46c-3620.
Coverage 1987-10-02 to 2022-10-31.

In [ ]:
!pip install clouddrift

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.5/134.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.7/698.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.8 MB/s eta 0:00:00
  Created

In [ ]:
# slide 21
from clouddrift.datasets import gdp1h
ds = gdp1h()
ds

<xarray.Dataset> Size: 16GB
Dimensions:                (traj: 19396, obs: 197214787)
Coordinates:
    id                     (traj) int64 155kB ...
    time                   (obs) datetime64[ns] 2GB ...
Dimensions without coordinates: traj, obs
Data variables: (12/58)
    BuoyTypeManufacturer   (traj) |S20 388kB ...
    BuoyTypeSensorArray    (traj) |S20 388kB ...
    CurrentProgram         (traj) float64 155kB ...
    DeployingCountry       (traj) |S20 388kB ...
    DeployingShip          (traj) |S20 388kB ...
    DeploymentComments     (traj) int16 39kB ...
    ...                     ...
    start_lat              (traj) float32 78kB ...
    start_lon              (traj) float32 78kB ...
    typebuoy               (traj) |S10 194kB ...
    typedeath              (traj) int8 19kB ...
    ve                     (obs) float32 789MB ...
    vn                     (obs) float32 789MB ...
Attributes: (12/18)
    Conventions:          CF-1.6
    acknowledgement:      Elipot, Shane; Sykulski, Adam; Lumpkin, Rick; Centu...
    contributor_name:     NOAA Global Drifter Program
    contributor_role:     Data Acquisition Center
    date_created:         2024-07-30T19:02:16.200531
    doi:                  10.25921/x46c-3620
    ...                   ...
    publisher_name:       GDP Drifter DAC
    publisher_url:        https://www.aoml.noaa.gov/phod/gdp
    summary:              Global Drifter Program hourly data
    time_coverage_end:    2022-10-31:00:00:00Z
    time_coverage_start:  1987-10-02:13:00:00Z
    title:                Global Drifter Program hourly drifting buoy collection

In [ ]:
west, east, south, north = 10, 45, -45, -20

ds = gdp1h().chunk({"obs": 10_000_000})
in_region = (
    (ds.lon >= west) & (ds.lon <= east)
    & (ds.lat >= south) & (ds.lat <= north)
)

print(f"Hourly records in R: {int(in_region.sum()):,}")

Hourly records in R: 3,848,969


In [ ]:
ds.typedeath.attrs

{'comments': '0 (buoy still alive), 1 (buoy ran aground), 2 (picked up by vessel), 3 (stop transmitting), 4 (sporadic transmissions), 5 (bad batteries), 6 (inactive status)',
 'long_name': 'Type of death',
 'units': '-'}

- 0: ("still alive", "censored"),    

still drifting; record ends only because the data release has **a cut-off date**

- ❗️**1: ("ran aground", "destination"),**❗️

flow carried it ashore = ***a real endpoint*** (eg: oil washing up on a beach)

- 2: ("picked up by vessel", "censored"),    

**removed by people**, unrelated to the flow; it would have kept drifting

- 3: ("stop transmitting", "censored"),    

**transmitter failed**, drifter still at sea ⚠️ near the coast it may actually have run aground

- 4: ("sporadic transmissions", "censored"),    

still drifting, but **positions** are patchy and **unreliable**

- 5: ("bad batteries", "censored"),    

**power ran out**, drifter still at sea

- 6: ("inactive status", "censored"),    

**flagged inactive**, whereabouts unknown

In [ ]:
import numpy as np

# slide 21: "Trajectory metadata and rowsize identify which observations belong to each drifter."
in_idx = np.nonzero(in_region.compute().values)[0]

bounds = np.cumsum(ds.rowsize.values)                        # end boundary of each drifter's block
traj_of_obs = np.searchsorted(bounds, in_idx, side="right")  # which drifter each record belongs to

print(f"Distinct trajectories in R: {len(np.unique(traj_of_obs)):,}")

Distinct trajectories in R: 1,149


In [ ]:
# slide 16 lists this as unresolved: "How many different drifters support each cell?"
cell = 1
n_cols = int(np.ceil((east - west) / cell))
n_rows = int(np.ceil((north - south) / cell))

lon_in = ds.lon[in_idx].values
lat_in = ds.lat[in_idx].values
col = np.floor((lon_in - west) / cell).astype(int)
row = np.floor((lat_in - south) / cell).astype(int)
cell_id = row * n_cols + col                         # (row, col) packed into one cell number

pairs = np.unique(cell_id * 100_000 + traj_of_obs)   # 100_000 > 19,396 trajectories, so no collisions
counts = np.bincount(pairs // 100_000, minlength=n_rows * n_cols)
occupied = counts[counts > 0]

print(f"Occupied cells: {len(occupied)} of {n_rows * n_cols}")
print(f"Median drifters per cell: {np.median(occupied):.0f}")
print(f"10th-90th percentile: {np.percentile(occupied, 10):.0f}-{np.percentile(occupied, 90):.0f}")
print(f"Cells with >= 10 drifters: {(occupied >= 10).sum()}")

Occupied cells: 649 of 875
Median drifters per cell: 71
10th-90th percentile: 24-138
Cells with >= 10 drifters: 614


#### Results: 1° grid, Agulhas box (10°E–45°E, 45°S–20°S)

- **Occupied cells: 649 of 875.** The box is 35 columns × 25 rows = 875 cells.
  The 226 empty cells are mostly land (South Africa, Mozambique, the south tip of Madagascar).
  Drifters cannot go there, so this is expected.
- **Median: 71 different drifters per cell.** A typical cell is visited by 71 different drifters.
- **10th–90th percentile: 24–138.** Coverage is uneven, but even the sparse cells have around 24 drifters.
- **614 cells have 10 or more drifters.** 35 occupied cells fall below this level.


In [ ]:
ds.attrs

{'Conventions': 'CF-1.6',
 'acknowledgement': 'Elipot, Shane; Sykulski, Adam; Lumpkin, Rick; Centurioni, Luca; Pazos, Mayra (2022). Hourly location, current velocity, and temperature collected from Global Drifter Program drifters world-wide. [indicate subset used]. NOAA National Centers for Environmental Information. Dataset. https://doi.org/10.25921/x46c-3620. Accessed [date]. Elipot et al. (2022): A Dataset of Hourly Sea Surface Temperature From Drifting Buoys, Scientific Data, 9, 567, https://dx.doi.org/10.1038/s41597-022-01670-2. Elipot et al. (2016): A global surface drifter dataset at hourly resolution, J. Geophys. Res.-Oceans, 121, https://dx.doi.org/10.1002/2016JC011716.',
 'contributor_name': 'NOAA Global Drifter Program',
 'contributor_role': 'Data Acquisition Center',
 'date_created': '2024-07-30T19:02:16.200531',
 'doi': '10.25921/x46c-3620',
 'history': 'version 2.01.1. Metadata from dirall.dat and deplog.dat',
 'institution': 'NOAA Atlantic Oceanographic and Meteorologica

In [ ]:
def count_region(ds, west, east, south, north):
    n_obs = ds.sizes["obs"]
    chunk = 10_000_000
    idx_list = []
    for start in range(0, n_obs, chunk):
        stop = min(start + chunk, n_obs)
        lon = ds.lon[start:stop].values
        lat = ds.lat[start:stop].values
        mask = (lon >= west) & (lon <= east) & (lat >= south) & (lat <= north)
        idx_list.append(np.nonzero(mask)[0] + start)
    idx = np.concatenate(idx_list)
    bounds = np.cumsum(ds.rowsize.values)
    n_traj = len(np.unique(np.searchsorted(bounds, idx, side="right")))
    return len(idx), n_traj

# Benguela: 0°–20°E, 38°S–15°S
print("Benguela:", count_region(ds, 0, 20, -38, -15))

Benguela: (2538266, 706)


### **Table 1.** Hourly records and distinct drifters in the chosen region and one rejected alternative.

| Box | Longitude | Latitude | Hourly records | Distinct drifters |
|---|---|---|---|---|
| **R (chosen)** | 10°E–45°E | 45°S–20°S | 3,848,969 | 1,149 |
| Benguela (rejected) | 0°–20°E | 38°S–15°S | 2,538,266 | 706 |

Both boxes use the boundaries given in the Week 1 illustration, and our pipeline reproduces
the published counts for each exactly, which validates the counting method.


---

### Why this is "enough"

Raw record counts overstate how much information the data contain, because one drifter can stay in the same cell for hundreds of hours. We therefore count the number of different drifters that visit each 1° cell. In R the median cell is visited by 71 different drifters (10th–90th percentile: 24–138), and 614 of 649 occupied cells are visited by at least 10. This suggests transition probabilities can be estimated from many independent realisations rather than from a handful of long tracks. These counts use a 1° grid to match the coverage summary in the Week 1 illustration; the transport model will start from 2° cells, which pools more drifters per cell and keeps the transition matrix better supported.

### Why Agulhas over Benguela
We compared both regions before committing. Benguela has 706 drifters against Agulhas's 1,149, and once we started thinking in terms of independent drifters per cell rather than raw records, that gap mattered more than it first looked. The bigger reason is what each region asks of a transition matrix. In Agulhas the retroflection splits material between two genuinely different outcomes, either back east into the Indian Ocean or west into the Atlantic, so the matrix has something interesting to estimate. Benguela's alongshore/offshore contrast felt like a thinner question to build a whole project around.